<a href="https://colab.research.google.com/github/BBqueenHack/aiot-environmental-anomaly-detection/blob/main/IoT_Project_Bianca_TestData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
import pandas as pd

# Reproducible results
rng = np.random.default_rng(42)

N = 3000

# Generate environmental readings
temperature = rng.uniform(10, 38, N)
humidity = rng.uniform(20, 90, N)

# Previous environmental baseline
baseline_temp = rng.uniform(20, 27, N)
baseline_hum = rng.uniform(40, 65, N)

# Calculate deviations
temp_deviation = temperature - baseline_temp
hum_deviation = humidity - baseline_hum

# Define anomaly labels
anomaly = (
    (temperature < 15)
    | (temperature > 30)
    | (humidity < 30)
    | (humidity > 75)
    | (np.abs(temp_deviation) > 3)
    | (np.abs(hum_deviation) > 15)
).astype(int)

# Create dataset
df = pd.DataFrame({
    "temperature": temperature,
    "humidity": humidity,
    "temp_deviation": temp_deviation,
    "hum_deviation": hum_deviation,
    "anomaly": anomaly
})

print("Dataset created successfully!")
print("Total samples:", len(df))
print("\nClass distribution:")
print(df["anomaly"].value_counts())

print("\nFirst 10 samples:")
print(df.head(10))

# Save dataset
df.to_csv("environmental_data.csv", index=False)


Dataset created successfully!
Total samples: 3000

Class distribution:
anomaly
1    2734
0     266
Name: count, dtype: int64

First 10 samples:
   temperature   humidity  temp_deviation  hum_deviation  anomaly
0    31.670769  78.045480        8.482435      16.204012        1
1    22.288596  61.586719        1.456590      15.435295        1
2    34.040742  37.139832       10.056528     -15.084270        1
3    29.526305  72.197250        8.431303      31.454320        1
4    12.636966  25.913663       -7.789743     -14.308342        1
5    37.317426  74.209248       13.826751      16.572918        1
6    31.311912  61.298966        6.724298      18.341625        1
7    32.009801  27.752522        6.030769     -25.942684        1
8    13.587182  25.565389      -11.892667     -31.834798        1
9    22.610806  72.558543       -3.389877      31.752679        1


In [3]:

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Prepare input features and labels
X = df[[
    "temperature",
    "humidity",
    "temp_deviation",
    "hum_deviation"
]].to_numpy(dtype=np.float32)

y = df["anomaly"].to_numpy(dtype=np.float32)

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Normalize features
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std = np.where(std == 0, 1, std)

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

# Build a small neural network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the model
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluate on unseen test data
loss, accuracy = model.evaluate(
    X_test_scaled, y_test, verbose=0
)

predictions = (
    model.predict(X_test_scaled, verbose=0).ravel() >= 0.5
).astype(int)

print("================================")
print("FLOAT32 MODEL RESULTS")
print("================================")
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test, predictions))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))
print()
model.summary()


FLOAT32 MODEL RESULTS
Training samples: 2400
Test samples: 600
Accuracy: 99.33 %

              precision    recall  f1-score   support

         0.0       0.98      0.94      0.96        53
         1.0       0.99      1.00      1.00       547

    accuracy                           0.99       600
   macro avg       0.99      0.97      0.98       600
weighted avg       0.99      0.99      0.99       600

Confusion matrix:
[[ 50   3]
 [  1 546]]



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 149 (600.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 100 (404.00 B)

In [4]:

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix

# Get trained model parameters
W1, b1 = model.layers[0].get_weights()
W2, b2 = model.layers[1].get_weights()

# Symmetric int8 quantization
def quantize_weights(weights):
    max_abs = np.max(np.abs(weights))
    scale = max_abs / 127 if max_abs > 0 else 1.0

    quantized = np.clip(
        np.round(weights / scale), -127, 127
    ).astype(np.int8)

    return quantized, scale

W1_q, scale1 = quantize_weights(W1)
W2_q, scale2 = quantize_weights(W2)

# Simulate quantized inference
def quantized_predict(X):
    W1_deq = W1_q.astype(np.float32) * scale1
    W2_deq = W2_q.astype(np.float32) * scale2

    hidden = np.maximum(0, X @ W1_deq + b1)
    logits = hidden @ W2_deq + b2

    return 1 / (1 + np.exp(-np.clip(logits, -50, 50)))

# Evaluate quantized model
q_probabilities = quantized_predict(X_test_scaled).ravel()
q_predictions = (q_probabilities >= 0.5).astype(int)

q_accuracy = accuracy_score(y_test, q_predictions)

# Parameter storage
float_size = (
    W1.nbytes + b1.nbytes +
    W2.nbytes + b2.nbytes
)

quantized_size = (
    W1_q.nbytes + b1.nbytes +
    W2_q.nbytes + b2.nbytes +
    8  # two float32 scales
)

print("================================")
print("INT8 QUANTIZATION RESULTS")
print("================================")

print("Float32 parameter size:", float_size, "bytes")
print("Quantized parameter size:", quantized_size, "bytes")
print("Memory reduction:",
      round((1 - quantized_size / float_size) * 100, 2), "%")

print()
print("Float32 accuracy:",
      round(accuracy_score(
          y_test,
          (model.predict(X_test_scaled, verbose=0).ravel() >= 0.5)
      ) * 100, 2), "%")

print("Quantized accuracy:", round(q_accuracy * 100, 2), "%")

print()
print("Quantization scales:", scale1, scale2)
print("Quantized confusion matrix:")
print(confusion_matrix(y_test, q_predictions))


INT8 QUANTIZATION RESULTS
Float32 parameter size: 196 bytes
Quantized parameter size: 84 bytes
Memory reduction: 57.14 %

Float32 accuracy: 99.33 %
Quantized accuracy: 99.33 %

Quantization scales: 0.037134383 0.042365868
Quantized confusion matrix:
[[ 50   3]
 [  1 546]]


In [5]:

import json
import numpy as np

# Export trained weights and normalization parameters
model_data = {
    "mean": mean.astype(float).tolist(),
    "std": std.astype(float).tolist(),

    "float32": {
        "W1": W1.tolist(),
        "b1": b1.tolist(),
        "W2": W2.tolist(),
        "b2": b2.tolist()
    },

    "int8": {
        "W1": W1_q.tolist(),
        "b1": b1.tolist(),
        "W2": W2_q.tolist(),
        "b2": b2.tolist(),
        "scale1": float(scale1),
        "scale2": float(scale2)
    }
}

# Save model to JSON
with open("model_data.json", "w") as f:
    json.dump(model_data, f, separators=(",", ":"))

print("Model exported successfully!")
print("File: model_data.json")

import os
print("File size:", os.path.getsize("model_data.json"), "bytes")

from google.colab import files
files.download("model_data.json")


Model exported successfully!
File: model_data.json
File size: 1619 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>